# Xwin-LM-70B-V0-1 + Wikipedia RAG

This notebook is a fork of the awesome [notebook](https://www.kaggle.com/code/simjeg/platypus2-70b-with-wikipedia-rag) by [@simjeg](https://www.kaggle.com/simjeg)

I modified it in three ways:
- I replaced the model with another model that has a more friendly Llama2 license
- I added a memory collection after each layer computation
- I added some tests to not run anything significant before the submission. Without these, the original code uses about 3 GPU hours.
- I removed the multithreading for loading layers
- I replaced "pertinent" by "relevant" in the prompt.


Other than that all the code and credit should go to @simjeg. Please upvote his notebook if you liked this one.

In [ ]:
# Installing offline dependencies
!pip install -U /kaggle/input/faiss-gpu-173-python310/faiss_gpu-1.7.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
!pip install -U /kaggle/input/transformers-432/transformers-4.32.1-py3-none-any.whl
!cp -rf /kaggle/input/sentence-transformers-222/sentence-transformers /kaggle/working/sentence-transformers
!pip install -U /kaggle/working/sentence-transformers
!pip install -U /kaggle/input/blingfire-018/blingfire-0.1.8-py3-none-any.whl

In [ ]:
import gc
import logging
from time import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import ctypes
from functools import partial

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# For RAG
import torch
import faiss
import blingfire as bf
from scipy.spatial.distance import cdist
from sentence_transformers import SentenceTransformer

# For Platypus2
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
from accelerate import init_empty_weights
from accelerate.utils.modeling import set_module_tensor_to_device
from safetensors.torch import load_file

def clean_memory():
    gc.collect()
    ctypes.CDLL("libc.so.6").malloc_trim(0)
    torch.cuda.empty_cache()
    
# Load data
# df = pd.read_csv("/kaggle/input/kaggle-llm-science-exam/test.csv", index_col="id")
df = pd.read_csv("/kaggle/input/mmlu-dataset-valid-only/valid_mmlu_1526_ind0.csv")
df.drop(columns=['Unnamed: 0'], inplace=True)

df.shape

In [ ]:
df['E'] = '' # dummy answer that allows us to preprocess the test datataset using functionality that works for the train set
df = df.replace(np.NaN, '')

df['A'] = df['A'].map(str)
df['B'] = df['B'].map(str)
df['C'] = df['C'].map(str)
df['D'] = df['D'].map(str)
df['E'] = df['E'].map(str)

df.reset_index(inplace=True, drop=True)

# Reorder columns
df = df[[col for col in df.columns if 'answer' not in col] + ['answer']]

df.head()

## 1. Wikipedia Retrieval Augmented Generation (RAG)

The following code is adapted from [the notebook of @MGöksu](https://www.kaggle.com/code/mgoksu/0-807-sharing-my-trained-with-context-model).

In [ ]:
if len(df) == 200: # Adedd by CPMP
    # Do nothing, this is interactive or saving
    df['context'] = "Wikipedia is a great source of information."
else:
    NUM_TITLES = 5
    NUM_SENTENCES_INCLUDE = 5
    FILTER_LEN = 100 
    MAX_SEQ_LEN = 512
    MODEL_PATH = '/kaggle/input/sentencetransformers-allminilml6v2/sentence-transformers_all-MiniLM-L6-v2' 
    # Idea : use bge-small-en instead (ranks 6 instead of 35 on the MTEB leaderboard)

    # Load embedding model
    start = time()
    print(f'Starting prompt embedding, t={time() - start :.1f}s')
    model = SentenceTransformer(MODEL_PATH, device='cuda:0').half()
    model.max_seq_length = MAX_SEQ_LEN

    # Get embeddings of prompts
    f = lambda row : " ".join([row['prompt'], row['A'], row['B'], row['C'], row['D'], row['E']])
    inputs = df.apply(f, axis=1).values # better results than prompt only
    prompt_embeddings = model.encode(inputs, normalize_embeddings=True, show_progress_bar=False)

    # Search closest sentences in the wikipedia index 
    print(f'Starting text search, t={time() - start :.1f}s')
    faiss_index = faiss.read_index("/kaggle/input/wikipedia-2023-07-faiss-index/wikipedia_202307.index")
    search_index = faiss_index.search(prompt_embeddings, NUM_TITLES)[1]

    # Free memory
    faiss_index.reset()
    del faiss_index
    clean_memory()

    # Load wikipedia index and filter it using the results of the search
    print(f'Starting loading texts, t={time() - start :.1f}s')
    wiki_data = pd.read_parquet("/kaggle/input/wikipedia-20230701/wiki_2023_index.parquet", columns=['id', 'file'])
    wiki_data = wiki_data.loc[np.unique(search_index)]

    # Retrieve text from wikipedia files and add it to wiki_data
    for filename in tqdm(wiki_data.file.unique()):
        # Load wikipedia file
        wiki_file = pd.read_parquet("/kaggle/input/wikipedia-20230701/" + filename, columns=['id', 'text'])
        wiki_file.set_index('id', inplace=True)

        # Add text column
        mask = wiki_data.file == filename
        indexes = wiki_data.loc[mask, 'id'].values
        wiki_data.loc[mask, 'text'] = wiki_file.loc[indexes, 'text'].values

        # Free memory
        del wiki_file
        clean_memory()

    # Split the texts into sentences and only keep NUM_SENTENCES_INCLUDE sentences
    print(f'Starting sentence retrieval t={time() - start :.1f}s')
    for idx in tqdm(range(len(df))):
        # Get question embedding
        q_embed = prompt_embeddings[idx]

        # Get sentence embeddings
        sentences = []
        for text in wiki_data.loc[search_index[idx], 'text']:
            if len(text) > 0:
                sentences += [text[start:end] for start, end in bf.text_to_sentences_and_offsets(text)[1] if (end - start) > FILTER_LEN]
        k_embed = model.encode(sentences, normalize_embeddings=True, show_progress_bar=False)

        # Get closest sentences
        distances = cdist(q_embed[None, :], k_embed)[0]
        knn = np.argsort(distances)[:NUM_SENTENCES_INCLUDE]
        context = "\n-".join(np.array(sentences)[knn])
        df.loc[idx, 'context'] = context

    # Free memory
    model.to('meta')
    del model, wiki_data, prompt_embeddings
    clean_memory()
    print(f'Context added, t={time() - start :.1f}s')

## 2: Run the 70B model

To such a large model on a single T4 GPU, we run it layer by layer and sample by sample

In [ ]:
# Create symlinks from kaggle datasets to fake cached model

checkpoint_path = Path("/root/.cache/")
checkpoint_path.mkdir(exist_ok=True, parents=True)

for part in [1, 2]:
    source_dir = Path(f'/kaggle/input/xwin-lm-70b-v0-1-part{part}')
    for path in source_dir.glob('*'):
        try:
            (checkpoint_path / path.name).symlink_to(path)
        except:
            pass

In [ ]:
# Class for sharded llama
MAX_LENGTH = 4096

class ShardedLlama:
    
    def __init__(self, checkpoint_path, device='cuda:0', dtype=torch.float16):
        """
        Sharded version of LlamaForCausalLM : the model is splitted into layer shards to reduce GPU memory usage.
        During the forward pass, the inputs are processed layer by layer, and the GPU memory is freed after each layer.
        To avoid loading the layers multiple times, we could save all the intermediate activations in RAM, but
        as Kaggle accelerators have more GPU memory than CPU, we simply batch the inputs and keep them on the GPU.

        Parameters
        ----------
        checkpoint_path : str or Path
            path to the checkpoint
        device : str, optional
            device, by default 'cuda:0'
        dtype : torch.dtype, optional
            dtype, by default torch.float16
        """
        
        # Save parameters
        self.checkpoint_path = Path(checkpoint_path)
        self.device = device 
        self.dtype = dtype

        # Create model
        self.config = AutoConfig.from_pretrained(self.checkpoint_path)
        # For flash attention when Turing architecture will be supported : https://github.com/Dao-AILab/flash-attention/issues/542
        # self.config.auto_map = {"AutoModelForCausalLM" : "togethercomputer/LLaMA-2-7B-32K--modeling_flash_llama.LlamaForCausalLM"} 
        
        self.tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = 'right'
        self.init_model()        
        self.layer_names = ['model.embed_tokens'] + [f'model.layers.{i}' for i in range(len(self.model.model.layers))] + ['model.norm', 'lm_head']
        
    def init_model(self):
                
        # Load meta model (no memory used)
        with init_empty_weights():
            self.model = AutoModelForCausalLM.from_config(self.config, trust_remote_code=True)
            self.model.tie_weights()
            
        self.layers = [self.model.model.embed_tokens] + list(self.model.model.layers) + [self.model.model.norm, self.model.lm_head]
            
        # Move buffers to device (not that much GPU memory used)
        for buffer_name, buffer in self.model.named_buffers():
            set_module_tensor_to_device(self.model, buffer_name, self.device, value=buffer, dtype=self.dtype)
       
    def load_layer(self, layer_name):
        state_dict = load_file(self.checkpoint_path / (layer_name + '.safetensors'), device=self.device)
        for param_name, param in state_dict.items():
            assert param.dtype != torch.int8, 'int8 not supported (need to add fp16_statistics)'
            set_module_tensor_to_device(self.model, param_name, self.device, value=param, dtype=self.dtype)
        
    def __call__(self, inputs, output_token):
        # inputs = [(prefix, suffix), ...] with prefix.shape[0] = 1 and suffix.shape[0] = 5
        
        # Reboot the model to make sure buffers are loaded and memory is clean
        del self.model
        clean_memory()
        self.init_model()
        
       # Send batch to device
        batch = [(prefix.to(self.device), suffix.to(self.device)) for prefix, suffix in inputs]
        n_suffixes = len(batch[0][1])
        suffix_eos = [(suffix != self.tokenizer.pad_token_id).sum(1) - 1 for _, suffix in inputs]

        # Create attention mask for the largest input, and position ids to use KV cache
        attention_mask = torch.finfo(self.dtype).min * torch.ones(MAX_LENGTH, MAX_LENGTH)
        attention_mask = attention_mask.triu(diagonal=1)[None, None, ...]
        attention_mask = attention_mask.to(self.device)
        position_ids = torch.arange(MAX_LENGTH, dtype=torch.long, device=self.device)[None, :]

        with ThreadPoolExecutor() as executor, torch.inference_mode():

            for i, (layer_name, layer) in enumerate(zip(self.layer_names, self.layers)):

                start = time()
                self.load_layer(self.layer_names[i])
                load_time = time() - start

                # Run layer
                for j, (prefix, suffix) in enumerate(batch):
                    if layer_name == 'model.embed_tokens':
                        batch[j] = (layer(prefix), layer(suffix))
                    elif layer_name == 'model.norm':
                        # Only keep the last hidden state at this point
                        batch[j] = (None, layer(suffix[torch.arange(n_suffixes), suffix_eos[j]][:, None]))
                    elif layer_name == 'lm_head':
                        batch[j] = (None, layer(suffix))
                    else:
                        # Run prefix
                        len_p, len_s = prefix.shape[1], suffix.shape[1]
                        new_prefix, (k_cache, v_cache) = layer(prefix, use_cache=True, attention_mask=attention_mask[:, :, -len_p:, -len_p:])
                        
                        # Run suffix
                        pos = position_ids[:, len_p:len_p + len_s].repeat(n_suffixes, 1)
                        attn = attention_mask[:, :, -len_s:, -len_p - len_s:].repeat(n_suffixes, 1, 1, 1)
                        kv_cache = (k_cache.repeat(n_suffixes, 1, 1, 1), v_cache.repeat(n_suffixes, 1, 1, 1))
                        new_suffix = layer(suffix, past_key_value=kv_cache, position_ids=pos, attention_mask=attn)[0]
                        batch[j] = (new_prefix, new_suffix)

                # Remove previous layer from memory (including buffers)
                layer.to('meta')
                clean_memory() # Added my CPMP to release memory after each layer is processed
                print(f'device {self.device}, {layer_name}, load time : {load_time:.1f}, run time: {time() - start - load_time:.1f}s')

        # Get scores
        batch = [suffix[:, 0, output_token].detach().cpu().numpy() for _, suffix in batch]
        
        return batch

In [ ]:
if len(df) == 200: # Adedd by CPMP
    # Do nothing, this is interactive or saving
    df['prediction'] = "A B C"
    df[['prediction']].to_csv('submission.csv')
else:
    # Run model on the 2 GPUs
    N_BATCHES = 5
    MAX_CONTEXT = 2500
    
    def get_tokens(row, tokenizer):
            system_prefix = "Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\n{instruction}\n\n### Input:\n{input_prefix}"
            instruction = f"Your task is to analyze the question and answer below. If the answer is correct, respond yes, if it is not correct respond no. As a potential aid to your answer, background context from Wikipedia articles is at your disposal, even if they might not always be relevant."
            input_prefix = f"Context: {row['context'][:MAX_CONTEXT]}\nQuestion: {row['prompt']}\nProposed answer: "
            prompt_prefix = system_prefix.format(instruction=instruction, input_prefix=input_prefix)
            prefix = tokenizer(prompt_prefix, return_tensors="pt", return_attention_mask=False, truncation=True, max_length=MAX_LENGTH)['input_ids']
            prompt_suffix = [f"{row[letter]}\n\n### Response:\n" for letter in 'ABCDE']
            suffix = tokenizer(prompt_suffix, return_tensors="pt", return_attention_mask=False, truncation=True, max_length=MAX_LENGTH, padding=True)['input_ids'][:, 1:]
            return prefix, suffix

    def run_model(device, df):
        model = ShardedLlama(checkpoint_path, device=f'cuda:{device}')
        f = partial(get_tokens, tokenizer=model.tokenizer)
        inputs = df.apply(f, axis=1).values
        batches = np.array_split(inputs, N_BATCHES)
        outputs = []
        for batch in batches:
            outputs += model(batch, output_token=4874)
        return outputs

    # Run model
    with ThreadPoolExecutor() as executor:
        outputs = list(executor.map(run_model, [0, 1], np.array_split(df, 2)))
        outputs = sum(outputs, [])

    # Save results
    n = len(df)
    for i, scores in enumerate(outputs):
        top3 = np.argsort(scores)[::-1]
        df.loc[i, 'prediction'] = ' '.join(['ABCDE'[j] for j in top3])
    df[['prediction']].to_csv('submission.csv')

    # Display performances if train set is used

    if 'answer' in df.columns:

        for i in range(n):
            df.loc[i, 'top_1'] = df.loc[i, 'prediction'][0]
            df.loc[i, 'top_2'] = df.loc[i, 'prediction'][2]        
            df.loc[i, 'top_3'] = df.loc[i, 'prediction'][4]

        top_i = [(df[f'top_{i}'] == df['answer']).sum() for i in [1, 2, 3]]
        print(f'top1 : {top_i[0]}/{n}, top2 : {top_i[1]}/{n}, top3 : {top_i[2]}/{n} (total={sum(top_i)} / {n})')
        print(f'Accuracy: {100*top_i[0]/n:.1f}%, map3: {100*(top_i[0] + top_i[1]*1/2 + top_i[2]*1/3).sum()/n:.1f}%')